<a href="https://colab.research.google.com/github/biglalo104/Projects/blob/main/Video%20Analytics%20Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
pip install opencv-python ultralytics numpy pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 3.3 MB/s eta 0:00:00


In [4]:
import cv2
import numpy as np
import time
import pandas as pd
from collections import defaultdict
from ultralytics import YOLO

class NairobiSafetyAnalytics:
    def __init__(self, video_path, output_path="output_video.mp4"):
        # Initialize YOLOv8 Nano model (use 'yolov8m.pt' for higher accuracy in production)
        self.model = YOLO('yolov8n.pt')

        # Video Capture & Output
        self.cap = cv2.VideoCapture(video_path)
        self.fps = int(self.cap.get(cv2.CAP_PROP_FPS))
        self.width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        self.out = cv2.VideoWriter(output_path, fourcc, self.fps, (self.width, self.height))

        # --- DEFINE REGIONS OF INTEREST (ROIs) ---
        # Polygon representing a pedestrian walkway/pavement (x,y coordinates)
        self.pavement_roi = np.array([[50, 300], [250, 300], [250, 600], [50, 600]], np.int32)
        # Polygon representing a Matatu Stage waiting zone
        self.matatu_stage_roi = np.array([[300, 100], [600, 100], [600, 400], [300, 400]], np.int32)

        # --- TRACKING & ANALYTICS VARIABLES ---
        self.crowd_density = 0
        self.boda_boda_violations = 0
        self.alerts_log = []

        # For Abandoned Object Detection: {track_id: {'first_seen': time, 'last_location': (x,y)}}
        self.object_tracker = defaultdict(lambda: {'first_seen': time.time(), 'last_location': (0,0)})
        self.ABANDONED_THRESHOLD_SEC = 5.0 # Alert if bag is stationary for 5 seconds

    def check_roi(self, point, roi_polygon):
        """Check if a point (center of bounding box) is inside a polygon ROI."""
        return cv2.pointPolygonTest(roi_polygon, point, False) >= 0

    def process_frame(self, frame):
        current_time = time.time()

        # 1. OBJECT DETECTION & TRACKING (YOLOv8)
        # Classes: 0=person, 3=motorcycle (boda boda), 24=suitcase, 26=handbag
        results = self.model.track(frame, persist=True, classes=[0, 3, 24, 26], verbose=False)

        self.crowd_density = 0 # Reset per frame for the specific ROI

        for r in results:
            boxes = r.boxes
            for box in boxes:
                cls_id = int(box.cls[0])
                cls_name = self.model.names[cls_id]
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                center_x, center_y = (x1 + x2) // 2, (y1 + y2) // 2
                center_point = (center_x, center_y)

                # Draw bounding boxes
                color = (0, 255, 0) if cls_name == 'person' else (255, 0, 0)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, f"{cls_name}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

                # --- ACTIVITY 1: Matatu Stage Crowd Dynamics ---
                if cls_name == 'person' and self.check_roi(center_point, self.matatu_stage_roi):
                    self.crowd_density += 1

                # --- ACTIVITY 2: Boda Boda Pavement Violation ---
                if cls_name == 'motorcycle' and self.check_roi(center_point, self.pavement_roi):
                    self.boda_boda_violations += 1
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
                    cv2.putText(frame, "ALERT: BODA BODA ON PAVEMENT", (x1, y1 - 20),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

                # --- ACTIVITY 3: Abandoned Object Detection ---
                if cls_name in ['suitcase', 'handbag'] and box.id is not None:
                    track_id = int(box.id[0])
                    prev_loc = self.object_tracker[track_id]['last_location']

                    # Calculate distance moved
                    dist = np.linalg.norm(np.array(center_point) - np.array(prev_loc))

                    if dist < 15: # Object is stationary (pixels threshold)
                        time_stationary = current_time - self.object_tracker[track_id]['first_seen']
                        if time_stationary > self.ABANDONED_THRESHOLD_SEC:
                            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 4)
                            cv2.putText(frame, "ALERT: ABANDONED OBJECT", (x1, y1 - 20),
                                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

                            # Log alert only once per object
                            if not self.object_tracker[track_id].get('alerted', False):
                                self.alerts_log.append({
                                    'Time': time.strftime('%Y-%m-%d %H:%M:%S'),
                                    'Type': 'Abandoned Object',
                                    'Location': f"X:{center_x}, Y:{center_y}"
                                })
                                self.object_tracker[track_id]['alerted'] = True
                    else:
                        # Object moved, reset timer
                        self.object_tracker[track_id] = {'first_seen': current_time, 'last_location': center_point}

                    self.object_tracker[track_id]['last_location'] = center_point

        # Draw ROIs on frame for visualization
        cv2.polylines(frame, [self.pavement_roi], True, (255, 255, 0), 2)
        cv2.putText(frame, "Pedestrian Zone", (self.pavement_roi[0][0], self.pavement_roi[0][1]-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

        cv2.polylines(frame, [self.matatu_stage_roi], True, (0, 255, 255), 2)
        cv2.putText(frame, "Matatu Stage", (self.matatu_stage_roi[0][0], self.matatu_stage_roi[0][1]-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

        # Dashboard Overlay
        cv2.rectangle(frame, (10, 10), (350, 120), (0, 0, 0), -1)
        cv2.putText(frame, f"Stage Crowd Density: {self.crowd_density}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        cv2.putText(frame, f"Boda Boda Violations: {self.boda_boda_violations}", (20, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        cv2.putText(frame, f"Security Alerts: {len(self.alerts_log)}", (20, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        return frame

    def run(self):
        print("Starting Video Analytics Pipeline... Press 'q' to quit.")
        while self.cap.isOpened():
            ret, frame = self.cap.read()
            if not ret:
                break

            processed_frame = self.process_frame(frame)
            self.out.write(processed_frame)

            cv2.imshow("Nairobi Public Safety Analytics", processed_frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        self.cap.release()
        self.out.release()
        cv2.destroyAllWindows()
        self.generate_report()

    def generate_report(self):
        """Generates a CSV report for the Command Center"""
        if self.alerts_log:
            df = pd.DataFrame(self.alerts_log)
            df.to_csv("security_alerts_report.csv", index=False)
            print(f"\n[SUCCESS] Processed video. {len(self.alerts_log)} security alerts logged to 'security_alerts_report.csv'.")
        else:
            print("\n[SUCCESS] Processed video. No abandoned object alerts triggered.")
        print(f"[INFO] Total Boda Boda pavement violations detected: {self.boda_boda_violations}")
        print("[INFO] Final output video saved.")

if __name__ == "__main__":
    # Replace 'nairobi_street.mp4' with your actual video file path
    VIDEO_INPUT = "nairobi_street.mp4"
    analytics = NairobiSafetyAnalytics(video_path=VIDEO_INPUT)
    analytics.run()

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Starting Video Analytics Pipeline... Press 'q' to quit.

[SUCCESS] Processed video. No abandoned object alerts triggered.
[INFO] Total Boda Boda pavement violations detected: 0
[INFO] Final output video saved.
